# Part 2: RFM Segmentation & Retention Strategy
## D2C Personal Care Brand – Customer Intelligence

**Objective:** Segment customers using RFM signals plus behavioural/support context, then design targeted retention actions for each segment.

**Snapshot date:** 2025-09-30  
**Prediction window:** Churn in next 60 days (label from `churn_labels.csv`)

---

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Consistent style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load all files
customers   = pd.read_csv('data/customers.csv')
orders      = pd.read_csv('data/orders.csv')
support     = pd.read_csv('data/support_tickets.csv')
web         = pd.read_csv('data/web_events_snapshot.csv')
churn       = pd.read_csv('data/churn_labels.csv')
intervene   = pd.read_csv('data/intervention_history.csv')
rfm_snap    = pd.read_csv('data/rfm_modeling_snapshot.csv')

print('Dataset shapes:')
for name, df in [('customers', customers), ('orders', orders), ('support', support),
                  ('web', web), ('churn', churn), ('interventions', intervene), ('rfm_snap', rfm_snap)]:
    print(f'  {name}: {df.shape}')

## 2. RFM Feature Construction

We use the pre-built `rfm_modeling_snapshot.csv` which already contains 180-day rolling features anchored to 2025-09-30. This prevents any look-ahead leakage.

- **Recency** = `recency_days` → days since last order as of snapshot
- **Frequency** = `frequency_180d` → number of orders in past 180 days
- **Monetary** = `monetary_180d` → gross spend in past 180 days (INR)

In [ ]:
df = rfm_snap.copy()

print('Recency (days since last order):')
print(df['recency_days'].describe().round(2))
print()
print('Frequency (orders in 180d):')
print(df['frequency_180d'].describe().round(2))
print()
print('Monetary (spend in 180d, INR):')
print(df['monetary_180d'].describe().round(2))

In [ ]:
# Score each dimension 1–5 (5 = best)
# Recency: lower days = more recent = higher score
df['R_score'] = pd.qcut(df['recency_days'], q=5, labels=[5,4,3,2,1]).astype(int)

# Frequency: higher orders = higher score; rank() breaks ties
df['F_score'] = pd.qcut(df['frequency_180d'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)

# Monetary: higher spend = higher score
df['M_score'] = pd.qcut(df['monetary_180d'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)

df['RFM_score'] = df['R_score'] + df['F_score'] + df['M_score']

print('RFM combined score distribution:')
print(df['RFM_score'].describe().round(2))
print()
print('Churn rate by RFM band:')
print(df.groupby('RFM_score')['churn_next_60d'].mean().round(3))

In [ ]:
# --- Chart 1: Churn rate by RFM score ---
churn_by_rfm = df.groupby('RFM_score')['churn_next_60d'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(churn_by_rfm['RFM_score'], churn_by_rfm['churn_next_60d'],
               color=plt.cm.RdYlGn(churn_by_rfm['RFM_score'] / 15))
ax.set_xlabel('Combined RFM Score (3–15)', fontsize=11)
ax.set_ylabel('Churn Rate (next 60d)', fontsize=11)
ax.set_title('Churn Rate drops sharply as RFM score rises', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xticks(churn_by_rfm['RFM_score'])
plt.tight_layout()
plt.savefig('charts/churn_by_rfm_score.png', bbox_inches='tight')
plt.show()

print('Customers with RFM ≤ 6 have >77% churn rate.')
print('Customers with RFM ≥ 13 have <12% churn rate.')

## 3. Segment Design Logic

Pure RFM scores group customers by transaction history but miss *why* they are at risk. We add four additional signals:

| Signal | Source column | Rationale |
|---|---|---|
| Avg discount % | `avg_discount_pct_180d` | Customers who only buy on steep discounts may not be genuinely loyal |
| Return rate | `return_rate_180d` | High returners create friction; if combined with high value, they're "unhappy" |
| Negative ticket rate | `negative_ticket_rate_90d` | Unresolved complaints predict churn independent of RFM |
| Web sessions 30d | `sessions_30d` | Low sessions + recent order drop = customer drifting away |

**Segment priority order** (rules evaluated top-down):
1. Champions
2. Loyal Customers
3. High-Value but Unhappy
4. At-Risk
5. Discount-Sensitive
6. Dormant
7. New Customers
8. Occasional Buyers (catch-all)

In [ ]:
def assign_segment(row):
    r, f, m      = row['R_score'], row['F_score'], row['M_score']
    rfm_s        = row['RFM_score']
    discount     = row['avg_discount_pct_180d']
    tickets      = row['ticket_count_90d']
    sessions     = row['sessions_30d']
    neg_t_rate   = row['negative_ticket_rate_90d']
    return_rate  = row['return_rate_180d']
    days_signup  = row['days_since_signup']

    # Champions: top-tier across all three RFM dimensions
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'

    # Loyal Customers: recent, frequent, decent combined score
    if r >= 3 and f >= 3 and rfm_s >= 10:
        return 'Loyal Customers'

    # High-Value but Unhappy: big spenders with product quality or service friction
    if m >= 4 and (return_rate > 0.2 or (tickets >= 1 and neg_t_rate > 0.5)):
        return 'High-Value but Unhappy'

    # At-Risk: formerly frequent/high-spend, now drifting (low recency score)
    if r <= 2 and (f >= 3 or m >= 3):
        return 'At-Risk'

    # Discount-Sensitive: high discount dependency, below-average RFM
    if discount > 0.35 and rfm_s <= 9:
        return 'Discount-Sensitive'

    # Dormant: very old recency, almost no web activity
    if r == 1 and sessions <= 2:
        return 'Dormant'

    # New Customers: signed up within 90 days, only 1 order so far
    if r >= 4 and f <= 2 and days_signup <= 90:
        return 'New Customers'

    return 'Occasional Buyers'


df['segment'] = df.apply(assign_segment, axis=1)

print('Segment sizes:')
print(df['segment'].value_counts())
print()
print('Churn rate per segment:')
print(df.groupby('segment')['churn_next_60d'].mean().round(3).sort_values(ascending=False))

In [ ]:
# --- Chart 2: Segment sizes ---
seg_counts = df['segment'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d62728','#ff7f0e','#e7a600','#bcbd22','#2ca02c','#17becf','#9467bd','#1f77b4']
bars = ax.barh(seg_counts.index, seg_counts.values, color=colors)
ax.bar_label(bars, padding=3)
ax.set_xlabel('Number of Customers', fontsize=11)
ax.set_title('Customer Count by Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/segment_sizes.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 3: Churn rate per segment ---
seg_churn = df.groupby('segment')['churn_next_60d'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
palette = ['#d62728' if v > 0.5 else '#ff7f0e' if v > 0.3 else '#2ca02c' for v in seg_churn.values]
bars = ax.bar(seg_churn.index, seg_churn.values, color=palette, edgecolor='white')
ax.bar_label(bars, fmt='%.0f%%', labels=[f'{v*100:.0f}%' for v in seg_churn.values], padding=2)
ax.set_ylabel('60-day Churn Rate', fontsize=11)
ax.set_title('Churn Rate by Segment', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('charts/churn_by_segment.png', bbox_inches='tight')
plt.show()

## 4. Non-RFM Signal Enrichment

We examine how the four additional signals behave across segments.

In [ ]:
signal_cols = ['avg_discount_pct_180d', 'return_rate_180d',
               'ticket_count_90d', 'sessions_30d', 'campaign_clicks_30d']

seg_signals = df.groupby('segment')[signal_cols].mean().round(3)
print('Average non-RFM signals per segment:')
print(seg_signals.to_string())

In [ ]:
# --- Chart 4: Heatmap of signal averages by segment ---
seg_order = ['Champions', 'Loyal Customers', 'New Customers',
             'Occasional Buyers', 'Discount-Sensitive',
             'High-Value but Unhappy', 'At-Risk', 'Dormant']

heat_data = seg_signals.loc[seg_order].T
heat_data.index = ['Avg Discount %', 'Return Rate', 'Tickets (90d)', 'Sessions (30d)', 'Campaign Clicks']

# Normalise each row to 0-1 for visual comparison
heat_norm = heat_data.div(heat_data.max(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(heat_norm, annot=heat_data.values, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Relative intensity'})
ax.set_title('Non-RFM Behavioural Signals across Segments', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
plt.savefig('charts/signal_heatmap.png', bbox_inches='tight')
plt.show()

print()
print('Key observations:')
print('  - Dormant customers have near-zero sessions AND near-zero discount usage (not price-driven).')
print('  - Discount-Sensitive avg discount = 44%, vs Champions at 28%.')
print('  - High-Value but Unhappy have 42% return rate vs 3-6% for loyal/champions.')
print('  - At-Risk customers show low web sessions (avg 3.1) despite past purchase history.')

In [ ]:
# --- Chart 5: RFM dimension boxplots per segment ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, label in zip(axes,
                           ['recency_days', 'frequency_180d', 'monetary_180d'],
                           ['Recency (days)', 'Frequency (180d)', 'Monetary (INR, 180d)']):
    order = df.groupby('segment')[col].median().sort_values().index
    sns.boxplot(data=df, x='segment', y=col, order=order, palette='coolwarm', ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
    ax.set_xlabel('')
    ax.set_ylabel(label)
    ax.set_title(label)

fig.suptitle('RFM Distribution by Segment', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('charts/rfm_boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 6: Scatter R vs M coloured by segment ---
fig, ax = plt.subplots(figsize=(9, 6))
palette_map = {
    'Champions': '#2ca02c',
    'Loyal Customers': '#1f77b4',
    'New Customers': '#17becf',
    'High-Value but Unhappy': '#e377c2',
    'At-Risk': '#ff7f0e',
    'Discount-Sensitive': '#bcbd22',
    'Dormant': '#d62728',
    'Occasional Buyers': '#9467bd'
}
for seg, grp in df.groupby('segment'):
    ax.scatter(grp['recency_days'], grp['monetary_180d'],
               label=seg, alpha=0.45, s=18, color=palette_map[seg])

ax.set_xlabel('Recency (days since last order)', fontsize=11)
ax.set_ylabel('Monetary Value – 180d (INR)', fontsize=11)
ax.set_title('Recency vs Monetary Value — coloured by segment', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('charts/recency_vs_monetary.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Chart 7: Discount % distribution for Discount-Sensitive vs Champions ---
fig, ax = plt.subplots(figsize=(8, 4))
for seg, col in [('Discount-Sensitive', '#e7a600'), ('Champions', '#2ca02c'), ('Dormant', '#d62728')]:
    sub = df[df['segment'] == seg]['avg_discount_pct_180d']
    ax.hist(sub, bins=20, alpha=0.6, label=f'{seg} (n={len(sub)})', color=col)

ax.set_xlabel('Avg Discount % (180d)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Discount Usage Distribution — Discount-Sensitive vs Champions vs Dormant', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('charts/discount_distribution.png', bbox_inches='tight')
plt.show()

disc_avg = df.groupby('segment')['avg_discount_pct_180d'].mean().sort_values(ascending=False)
print('Avg discount % by segment:')
print(disc_avg.round(3))

## 5. Segment Profiles — Summary Table

In [ ]:
profile_cols = [
    'recency_days', 'frequency_180d', 'monetary_180d',
    'avg_discount_pct_180d', 'return_rate_180d',
    'ticket_count_90d', 'sessions_30d', 'churn_next_60d'
]
display_names = [
    'Recency (days)', 'Freq (180d)', 'Spend INR (180d)',
    'Avg Discount', 'Return Rate',
    'Tickets (90d)', 'Sessions (30d)', 'Churn Rate'
]

profile = df.groupby('segment')[profile_cols].mean().round(3)
profile.columns = display_names
profile = profile.loc[seg_order]

# Add count column
profile.insert(0, 'Count', df['segment'].value_counts().reindex(seg_order))

print('Full segment profile:')
print(profile.to_string())

## 6. Segment Summary Observations

### Champions (n=401, churn=10%)
High R/F/M across the board. Sessions avg 9.8/month. Only 10% churn — these customers are engaged and happy. Any intervention risks feeling intrusive.

### Loyal Customers (n=499, churn=22%)
Recent, frequent, good spend. Slightly below Champion threshold. Their 22% churn is mostly driven by customers who had a recent dip in recency (R_score=3). Loyalty reinforcement works here.

### High-Value but Unhappy (n=47, churn=70%)
Average spend ₹2,350/180d — top-tier — but return rate is 42% and negative ticket rate is high. These 47 customers represent significant revenue at risk. The issue is product fit or quality, not price.

### At-Risk (n=462, churn=74%)
Previously bought frequently/spent well, but recency dropped sharply (avg 130 days). Web sessions avg 3.1 vs 9.8 for Champions. The drop in engagement likely preceded the churn decision.

### Discount-Sensitive (n=202, churn=55%)
Avg discount 44%. These customers activate only when they get deals. Without a discount trigger, they go quiet. Low frequency (avg 1.1 orders/180d) despite relatively recent purchases.

### Dormant (n=203, churn=91%)
Recency avg 292 days. Near-zero sessions. Near-zero discount usage — they didn't even respond to existing campaigns. Reactivation cost likely exceeds expected return for most of this group.

### New Customers (n=96, churn=17%)
Signed up within 90 days, only 1–2 orders. Low churn so far but uncertain loyalty. Early onboarding interventions matter here.

### Occasional Buyers (n=490, churn=59%)
Mid-range across everything. Neither loyal nor gone. Hard to win over with generic messages — need interest-specific nudges.

In [ ]:
# Save segments.csv
seg_cols = ['customer_id', 'segment', 'R_score', 'F_score', 'M_score', 'RFM_score',
            'recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d',
            'avg_discount_pct_180d', 'ticket_count_90d', 'negative_ticket_rate_90d',
            'sessions_30d', 'last_visit_days_ago', 'campaign_clicks_30d', 'churn_next_60d']
df[seg_cols].to_csv('segments.csv', index=False)
print(f'segments.csv saved with {len(df)} rows and {len(seg_cols)} columns.')
print()
print('Preview:')
print(df[seg_cols].head(5).to_string())